In [ ]:
import sys
sys.path.append("../..")

In [ ]:
from artseek.data.datasets.explain_me import ExplainMeDataset
from artseek.data.datasets.artpedia import ArtpediaDataset
from PIL import Image
from transformers import AutoProcessor, CLIPModel
import torch
import lovely_tensors as lt
from tqdm import tqdm
import torch.nn.functional as F
lt.monkey_patch()

In [ ]:
device = torch.device("cuda")

In [ ]:
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model.to(device)

In [ ]:
ds = ArtpediaDataset("../../data/external/artpedia", split="test")

In [ ]:
t = torch.zeros((len(ds), 512))
t

In [ ]:
# read images test1.png test2.png test3.png from directory data/images, they are not in the dataset
images = [Image.open(f"../../data/images/test{i}.png") for i in range(1, 5)]

In [ ]:
for i in tqdm(range(len(ds))):
    ds_image = ds[i]["image"]
    ds_inputs = processor(images=ds_image, return_tensors="pt", padding=True).to(device)
    ds_outputs = model.get_image_features(**ds_inputs)[0]
    t[i] = ds_outputs

In [ ]:
inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
outputs = model.get_image_features(**inputs)

In [ ]:
outputs

In [ ]:
t1_norm = F.normalize(outputs, p=2, dim=1)
t2_norm = F.normalize(t, p=2, dim=1)

In [ ]:
cos_sim = torch.matmul(t1_norm.to(device), t2_norm.T.to(device))

# Get top 3 indices with highest similarity for each row of t1
topk_values, topk_indices = torch.topk(cos_sim, k=3, dim=1)

print("Top 3 most similar indices in t2 for each row in t1:")
print(topk_indices)

In [ ]:
topk_indices.v

In [ ]:
ds[142]["image"]